# Quick exploration into `everef` data

Scratch pad for configuring `dlt` pipeline with `everef` data

### Imports

In [1]:
import dlt
from dlt.sources.filesystem import filesystem, read_csv
from dlt.extract import DltResource

from datetime import date, datetime, timedelta, UTC

from collections.abc import Generator, Iterator

# import requests
from dlt.sources.helpers import requests
from requests import codes

import logging.config

from email.utils import parsedate_to_datetime

import pandas as pd

### Sources

In [2]:
BASE_URL = "https://data.everef.net/market-history"
destination = "file://../.local/notebook/data/bronze"

totals_file = f"{BASE_URL}/totals.json"

In [3]:
import pathlib
import json
import atexit

# https://www.youtube.com/watch?v=9L77QExPmI0
logger = logging.getLogger("eve_market_dlt")
logging_config = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "simple": {
            "format": "%(levelname)s: %(message)s"
        },
        "detailed": {
            "format": "[%(levelname)s] %(asctime)s | %(module)s:L%(lineno)d : %(message)s",
            "datefmt": "%Y-%m-%dT%H:%M:%S%z"
        }
    },
    "handlers": {
        "stderr": {
            "class": "logging.StreamHandler",
            "level": "WARNING",
            "formatter": "detailed",
            "stream": "ext://sys.stderr"
        },
        # "file": {
        #     "class": "logging.handlers.RotatingFileHandler",
        #     "level": "DEBUG",
        #     "formatter": "simple",
        #     "filename": "logs/tmp.log",
        #     "maxBytes": 10000,
        #     "backupCount": 3
        # },
        "queue_handler": {
            "class": "logging.handlers.QueueHandler",
            "respect_handler_level": True,
            "handlers": [
                "stderr",
                # "file"
            ]
        }
    },
    "loggers": {
        "root": {
            "level": "DEBUG",
            "handlers": [
                "queue_handler"
            ]
        }
    }
}

def setup_logging():
    # config_file = pathlib.Path("logging_configs/config.json")
    # with open(config_file) as f_in:
    #     config = json.load(f_in)
    # logging.config.dictConfig(config)
    logging.config.dictConfig(logging_config)
    queue_handler = logging.getHandlerByName("queue_handler")
    if queue_handler is not None:
        queue_handler.listener.start()
        atexit.register(queue_handler.listener.stop)

setup_logging()

In [4]:
def get_date(filename: str) -> date:
    date_str = filename.removeprefix("market-history-") \
                        .removesuffix(".csv.bz2")
    return date.fromisoformat(date_str)

def get_everef_file_url(curr_date: date) -> str:
    return f"{BASE_URL}/{curr_date.year}/market-history-{curr_date.isoformat()}.csv.bz2"

def daterange(date_start: date, date_end: date) -> Generator[date]:
    for n in range((date_end - date_start).days + 1):
        yield date_start + timedelta(n)

In [5]:
url = get_everef_file_url(get_date("2026-04-23"))
response = requests.head(url)

response.status_code == codes.OK

True

In [6]:
def validate_content_length_header(item: dict[str, str], response: requests.Response) -> None:
    content_length = response.headers.get("content-length")
    if content_length is None:
        logger.warning("Everef is missing content-length header for %s", item["url"])
        return
    
    if content_length and int(content_length) == 0:
        logger.warning("Everef file has content-length of 0: %s", item["url"])
    else:
        item.update({"content_length": int(content_length)})

def validate_last_modified_header(item: dict[str, str], response: requests.Response) -> None:
    last_modified = response.headers.get("last-modified")
    if last_modified is None:
        logger.warning("Everef is missing last-modified header for %s", item["url"])
        return
    
    try:
        last_modified_dt = parsedate_to_datetime(last_modified)
    except ValueError:
        logger.warning(
            "Everef returned invalid last-modified=%r for %s",
            last_modified,
            item["url"]
        )

    last_modified_iso = last_modified_dt.astimezone(UTC).isoformat()
    item.update({
        "last_modified": last_modified_iso
    })


In [7]:
@dlt.resource(name="market_history_urls", selected=False)
def list_file_urls(date_start: date, date_end: date) -> Iterator[dict[str, str]]:
    for curr_date in daterange(date_start, date_end):
        yield {
            "market_date": curr_date.isoformat(), 
            "url": get_everef_file_url(curr_date)
        }

EVEREF_PROBE_CLIENT = requests.Client(raise_for_status=False,
                                      status_codes=(429, 500, 502, 503, 504))

@dlt.transformer(name="market_history_files", selected=False, parallelized=True)
def probe_url_file(item: dict[str, str]) -> Iterator[dict[str, int | str]]:
    try:
        response = EVEREF_PROBE_CLIENT.head(item["url"], allow_redirects=True)
    except requests.RequestException as e:
        logger.warning("Everef probe failed at %s: %s", item["url"], e)
        return
    if response.status_code == 404:
        logger.warning("Everef file missing for %s, %s", item["market_date"], item["url"])
        return
    if response.status_code >= 400:
        logger.warning(
            "Unexpected Everef status HTTP %s for %s",
            response.status_code,
            item["url"]
        )
        return
    
    validate_content_length_header(item, response)
    validate_last_modified_header(item, response)

    yield item

@dlt.transformer(name="market_history",
                 parallelized=True,
                #  file_format="parquet",
                 write_disposition="merge",
                 primary_key=["date", "region_id", "type_id"])
def read_market_history_csv(item: dict[str, int | str]) -> Iterator[pd.DataFrame]:
    file_url, market_date = item["url"], item["market_date"]
    ingested_at = datetime.now(UTC).isoformat()

    try:
        chunks = pd.read_csv(
            file_url,
            compression="bz2",
            chunksize=20_000,
            # dtype_backend="pyarrow"
        )
        
        for chunk in chunks:
            chunk["_source_market_date"] = market_date
            chunk["_ingested_at"] = ingested_at
            yield chunk

    except Exception as e:
        logger.warning("Could not read Everef CSV %s: %s", file_url, e)
        return
    
@dlt.source(name="everef")
def everef_source():
    return list_file_urls(get_date("2025-01-01"), get_date("2025-01-31")) \
            | probe_url_file \
            | read_market_history_csv
    

# @dlt.source(name="everef")
# def everef_source(year: int, file_glob: str) -> DltResource:
#     files = filesystem(
#         bucket_url=f"{BASE_URL}/{year}/",
#         file_glob=file_glob
#     )
    
#     market_history = (
#         files
#         | read_csv(
#             chunksize=20_000,
#             compression="bz2"
#         )
#     ).with_name("market_history")

#     market_history.apply_hints(
#         primary_key=["region_id", "type_id", "date"],
#         write_disposition="merge"
#     )

#     return market_history

In [8]:
pipeline = dlt.pipeline(
    pipeline_name="everef_pipeline_dev",
    # destination="filesystem",
    destination="duckdb",
    dataset_name="everef_history_dev",
    dev_mode=True
)


In [9]:
load_info = pipeline.run(
    everef_source()
)



2026-05-06 02:43:28,910|[WARNING]|77254|137874120906560|dlt|extractors.py|_compute_tables:547|In resource: market_history, when merging arrow schema with dlt schema, several column hints were different. dlt schema hints were kept and arrow schema and data were unmodified. It is up to destination to coerce the differences when loading. Change log level to INFO for more details.
2026-05-06 02:43:29,047|[WARNING]|77254|137874120906560|dlt|extractors.py|_compute_tables:547|In resource: market_history, when merging arrow schema with dlt schema, several column hints were different. dlt schema hints were kept and arrow schema and data were unmodified. It is up to destination to coerce the differences when loading. Change log level to INFO for more details.
2026-05-06 02:43:29,065|[WARNING]|77254|137874120906560|dlt|extractors.py|_compute_tables:547|In resource: market_history, when merging arrow schema with dlt schema, several column hints were different. dlt schema hints were kept and arrow 

In [10]:
print(load_info)

Pipeline everef_pipeline_dev load step completed in 4.33 seconds
1 load package(s) were loaded to destination duckdb and into dataset everef_history_dev_20260506074328
The duckdb destination used duckdb:////home/rnoh/dev/eve-market/experiments/everef_pipeline_dev.duckdb location to store data
Load package 1778053408.4387803 is LOADED and contains no failed jobs


In [11]:
print(pipeline.default_schema.to_pretty_yaml())


version: 2
version_hash: /m8RvQF1f6XgDw+Bm6vzZ21I9FnuKOsmGjkIb6ofrDM=
engine_version: 11
name: everef
tables:
  _dlt_version:
    columns:
      version:
        data_type: bigint
        nullable: false
      engine_version:
        data_type: bigint
        nullable: false
      inserted_at:
        data_type: timestamp
        nullable: false
      schema_name:
        data_type: text
        nullable: false
      version_hash:
        data_type: text
        nullable: false
      schema:
        data_type: text
        nullable: false
    write_disposition: skip
    resource: _dlt_version
    description: Created by DLT. Tracks schema updates
  _dlt_loads:
    columns:
      load_id:
        data_type: text
        nullable: false
        precision: 64
      schema_name:
        data_type: text
        nullable: true
      status:
        data_type: bigint
        nullable: false
      inserted_at:
        data_type: timestamp
        nullable: false
      schema_version_hash:
    

In [12]:
table = pipeline.default_schema.tables["market_history"]

for col in ["date", "region_id", "type_id"]:
    print(col, table["columns"][col])
    

date {'name': 'date', 'nullable': False, 'data_type': 'text', 'primary_key': True}
region_id {'name': 'region_id', 'nullable': False, 'data_type': 'bigint', 'primary_key': True}
type_id {'name': 'type_id', 'nullable': False, 'data_type': 'bigint', 'primary_key': True}


In [13]:
file_url = "market-history-2026-04-17.csv.bz2"
market_date = "2026-04-17"

In [14]:
import pyarrow as pa

chunks = pd.read_csv(
    file_url,
    compression="bz2",
    chunksize=20_000,
)

chunk = next(chunks)
chunk["_source_market_date"] = market_date
chunk["_ingested_at"] = datetime.now(UTC).isoformat()

print(chunk.dtypes)
print(chunk[["date", "region_id", "type_id"]].isna().sum())

arrow_table = pa.Table.from_pandas(chunk, preserve_index=False)
print(arrow_table.schema)

average                float64
date                       str
highest                float64
lowest                 float64
order_count              int64
volume                   int64
http_last_modified         str
region_id                int64
type_id                  int64
_source_market_date        str
_ingested_at               str
dtype: object
date         0
region_id    0
type_id      0
dtype: int64
average: double
date: large_string
highest: double
lowest: double
order_count: int64
volume: int64
http_last_modified: large_string
region_id: int64
type_id: int64
_source_market_date: large_string
_ingested_at: large_string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1380
